# TP 2 — Construire l'auto-attention pas à pas

Boucles (moyenne causale) → produit matriciel (poids fixes) → masque + softmax → self-attention (poids appris).

**Invariant** : chaque ligne de poids donne la contribution des tokens passés au token courant, et somme à 1.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Agréger par produit matriciel

In [2]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)

b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)
print("somme des lignes de a :", a.sum(1))

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
somme des lignes de a : tensor([1., 1., 1.])


### Réponses
- **c₂** (2ᵉ ligne) = ½·b₁ + ½·b₂ : la moyenne des deux premières lignes de `b`. Avec `b = [[2,7],[6,4],[6,5]]` (seed 42) : c₂ = ((2+6)/2, (7+4)/2) = **(4, 5,5)** ✔ (cf. sortie).
- **Somme des lignes = 1** : on a divisé chaque ligne du triangle par son nombre de 1 (`a.sum(1, keepdim=True)`). Chaque `cₜ` est donc une **moyenne** (combinaison convexe) des `bᵢ` visibles, pas une somme qui grossirait avec t.

## Étape 2 — Moyenner uniquement le passé (version boucles)

In [3]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]               # (t+1, C)
        xbow[b, t] = torch.mean(xprev, dim=0)

for t in (0, 3, 7):
    print(f"t={t} : xprev.shape =", tuple(x[0, :t + 1].shape))
print(x[0]); print(xbow[0])

t=0 : xprev.shape = (1, 2)
t=3 : xprev.shape = (4, 2)
t=7 : xprev.shape = (8, 2)
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### Réponses
- Formes de `xprev` : **t=0 → (1, 2)**, **t=3 → (4, 2)**, **t=7 → (8, 2)**.
- **`dim=0`** : l'axe 0 de `xprev` est l'axe **temporel** (les tokens visibles). On moyenne à travers les tokens, en gardant les C features → résultat de forme `(C,)`.

## Étapes 3–4 — Vectoriser

In [4]:
# Version 2 : matrice triangulaire normalisée
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x            # (T,T) @ (B,T,C) -> (B,T,C)
print(torch.allclose(xbow, xbow2, atol=1e-6))

# Version 3 : masque -inf + softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3, atol=1e-6))
print(wei)

True
True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


### Réponses
- **`(T,T) @ (B,T,C)`** : `@` diffuse (broadcast) sur les dimensions de tête. `wei` est traité comme `(1,T,T)` → étendu en `(B,T,T)`, puis un produit matriciel `(T,T)@(T,C)` est fait pour chaque batch → `(B,T,C)`.
- **Softmax sur −∞** : exp(−∞) = 0, donc ces positions reçoivent un poids **exactement nul**. Les zéros restants (positions visibles) donnent exp(0)=1 partout puis normalisation → 1/(t+1) : la même moyenne uniforme. L'intérêt : on pourra remplacer les zéros par des **scores appris**.

## Étape 5 — Calculer des poids d'attention (une tête)

In [5]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                                   # (B,T,16)
q = query(x)                                 # (B,T,16)
wei = q @ k.transpose(-2, -1)                # (B,T,16)@(B,16,T) -> (B,T,T)
wei = wei * head_size**-0.5                  # mise à l'échelle 1/sqrt(dh)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)                                 # (B,T,16)
out = wei @ v                                # (B,T,T)@(B,T,16) -> (B,T,16)

print("k,q,v :", k.shape, "| wei :", wei.shape, "| out :", out.shape)
torch.set_printoptions(precision=2, sci_mode=False)
print(wei[0])

k,q,v : torch.Size([4, 8, 16]) | wei : torch.Size([4, 8, 8]) | out : torch.Size([4, 8, 16])
tensor([[1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.40, 0.60, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.31, 0.29, 0.40, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.32, 0.22, 0.24, 0.21, 0.00, 0.00, 0.00, 0.00],
        [0.15, 0.20, 0.17, 0.15, 0.34, 0.00, 0.00, 0.00],
        [0.13, 0.25, 0.13, 0.11, 0.31, 0.07, 0.00, 0.00],
        [0.16, 0.20, 0.11, 0.11, 0.14, 0.17, 0.11, 0.00],
        [0.08, 0.12, 0.11, 0.15, 0.11, 0.11, 0.16, 0.16]],
       grad_fn=<SelectBackward0>)


In [6]:
# Vérifications de l'invariant
print("lignes somment à 1 :", torch.allclose(wei.sum(-1), torch.ones(B, T)))
print("aucun poids vers le futur :", (wei.masked_select(torch.triu(torch.ones(T, T), 1).bool().expand(B, T, T)) == 0).all().item())

lignes somment à 1 : True
aucun poids vers le futur : True


### Réponses
- **Dernier token** : c'est la **dernière ligne** `wei[0, -1]` (t = 7). C'est la seule ligne sans aucun zéro : il voit les 8 tokens, avec des poids désormais **non uniformes** et dépendants des données.
- **Pourquoi mélanger `v` et pas `k` ?** Les rôles sont séparés :
  - **Q** (query) : « ce que je cherche » ;
  - **K** (key) : « ce que je contiens, pour être trouvé » → sert uniquement à calculer la compatibilité `q·k` ;
  - **V** (value) : « ce que je transmets si on me choisit ».

  Utiliser `k` comme contenu forcerait le même vecteur à servir à la fois d'étiquette de recherche et d'information transmise. Avec `V`, le modèle peut apprendre une projection dédiée au message.

## Étape 6 — Stabiliser softmax

In [7]:
torch.manual_seed(1337)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5
print("var(raw) =", raw.var().item(), " var(scaled) =", scaled.var().item())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

var(raw) = 18.44363021850586  var(scaled) = 1.1527268886566162
tensor([0.19, 0.14, 0.24, 0.14, 0.29])
tensor([0.03, 0.00, 0.16, 0.00, 0.80])


### Bilan du TP 2
- Var(q·k) ≈ d_h (somme de d_h produits de variance 1) → ici ≈ 16. Après division par √d_h on revient à ≈ 1.
- Sans ça, softmax reçoit de grands écarts de scores et tend vers un **one-hot** (cf. `z*8`) : chaque token n'écoute qu'un seul autre et les gradients deviennent quasi nuls. À l'initialisation, on veut des poids **diffus**.
- ✔ lignes somment à 1, ✔ pas de poids vers le futur, ✔ forme finale `(B, T, d_h)`.
- **Formes** : B = séquences indépendantes, T = positions, C = dimension d'embedding, d_h = `head_size` (dimension de Q/K/V d'une tête).
- **Note** : c'est de l'auto-attention *causale* (décodeur). Sans le masque on aurait un encodeur (tous les tokens se voient). Les tokens des différents batchs ne communiquent jamais.